# Clean segment variables with GraphML explicit road classes and diagnostics

This notebook builds a segment level modeling table from the midpoint split movement result.

Updates in this version:

1. Road class variables are read from the actual GraphML fields such as `road_class_en`, `EN_Class`, `highway`, `road_class_cn`, and `CN_Class`.
2. Existing long running graph results are reused. By default only road class columns are refreshed from the current GraphML.
3. Betweenness and closeness approximations have visible progress bars.
4. POI inputs can be shapefile, GeoPackage, or CSV with coordinates or WKT geometry.
5. Final outputs include a variable dictionary and VIF table.


This version uses explicit GraphML road-class and POI-category variable lists. It also adds a final correlation/VIF control block and a copy-paste model variable block.


In [2]:
# =========================================================
# 0. Imports and controls
#    Python 3.8 compatible
# =========================================================
import ast
import json
import math
import os
import pickle
import re
from collections import Counter, OrderedDict, defaultdict
from pathlib import Path

import geopandas as gpd
import networkx as nx
import numpy as np
import pandas as pd
import pyproj
from shapely import wkt
from shapely.geometry import LineString, MultiLineString, Point, box
from shapely.ops import linemerge, unary_union
from shapely.strtree import STRtree
from tqdm import tqdm

try:
    import osmnx as ox
except Exception as exc:
    ox = None
    print("Warning: osmnx import failed:", exc)

# -------------------------
# Resume controls
# -------------------------
RESUME_FROM_EXISTING = True

# Keep expensive stages unless they are missing.
FORCE_REBUILD_RAW_EVENTS = False
FORCE_REBUILD_RIDER_FEATURES = False
FORCE_REBUILD_GRAPH_CORE = False
FORCE_REFRESH_ROAD_CLASS = True
FORCE_REBUILD_LANDUSE_FEATURES = False
FORCE_REBUILD_SEGMENT_FEATURES = False
FORCE_REBUILD_VIF = False

# Centrality controls. If previous centrality columns exist in cache, they are reused.
COMPUTE_BETWEENNESS_IF_MISSING = True
COMPUTE_CLOSENESS_IF_MISSING = True
BETWEENNESS_K = 300
BETWEENNESS_BATCH_SIZE = 20
FARNESS_SAMPLE_SOURCES = 200

# Output controls.
WRITE_CSV_COPY = True
WRITE_GPKG = True
WRITE_SHP = True

# Modeling controls.
BUFFER_M = 300
OVERSPEED_THRESHOLD_KMH = 20.0
LONG_SEGMENT_THRESHOLD_M = 2000.0
TZ_SHIFT_HOURS = 8
RANDOM_SEED = 42

# VIF controls.
VIF_MAX_ROWS = 200000
VIF_MIN_NON_NULL_SHARE = 0.80
VIF_CORR_DROP_THRESHOLD = 0.995

np.random.seed(RANDOM_SEED)
os.environ.setdefault("SHAPE_RESTORE_SHX", "YES")


'YES'

In [3]:
# =========================================================
# 1. Paths
# =========================================================
try:
    BASE_DIR = Path(__file__).resolve().parent
except NameError:
    BASE_DIR = Path.cwd()

OUTPUT_ROOT = BASE_DIR / "outputs_new_clear" / "segment_variables_clean"
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

SEGMENT_CSV_CANDIDATES = [
    BASE_DIR / "outputs_new_clear" / "OD" / "MidPoints" / "courier_final_distances_filtered.csv",
    BASE_DIR / "outputs_new_clear" / "OD" / "MidPoints" / "courier_final_distances_segments_virtual_edge.csv",
    BASE_DIR / "courier_final_distances_filtered.csv",
]
GRAPHML_CANDIDATES = [
    BASE_DIR / "baoding_clear.graphml",
    BASE_DIR / "baoding_dense_30.graphml",
]

WAYBILL_CSV = BASE_DIR / "all_waybill_info.csv"
WAVE_CSV = BASE_DIR / "courier_wave_info.csv"

# POI candidates. Add your own POI table here if the auto search misses it.
POI_INPUT_CANDIDATES = [
    BASE_DIR / "baoding" / "保定数据" / "POI_保定.shp",
    BASE_DIR / "POI_保定.shp",
    BASE_DIR / "poi.shp",
    BASE_DIR / "poi.csv",
]
REST_INPUT_CANDIDATES = [
    BASE_DIR / "baoding" / "保定数据" / "大众点评餐馆_保定.shp",
    BASE_DIR / "大众点评餐馆_保定.shp",
    BASE_DIR / "restaurant.shp",
    BASE_DIR / "restaurants.csv",
]

RAW_EVENT_STATE_CSV = OUTPUT_ROOT / "rider" / "raw_event_state.csv.gz"
RIDER_FEATURE_CSV = OUTPUT_ROOT / "rider" / "rider_features.csv"

GRAPH_DIR = OUTPUT_ROOT / "graph"
GRAPH_DIR.mkdir(parents=True, exist_ok=True)
GRAPH_CORE_CACHE_PKL = GRAPH_DIR / "graph_core_cache.pkl"
GRAPH_FEATURE_PKL = GRAPH_DIR / "graph_feature_cache.pkl"       # backward compatible name
GRAPHML_DIAGNOSTIC_JSON = GRAPH_DIR / "graphml_attribute_diagnostics.json"
GRAPHML_ROAD_CLASS_CSV = GRAPH_DIR / "graphml_road_class_values.csv"
CONEDGE_FEATURE_CSV = GRAPH_DIR / "conedge_features.csv.gz"
CONEDGE_GPKG = GRAPH_DIR / "conedge_features.gpkg"

LANDUSE_DIR = OUTPUT_ROOT / "landuse"
LANDUSE_CONEDGE_CSV = LANDUSE_DIR / "conedge_landuse_300m.csv.gz"
POI_CATEGORY_CSV = LANDUSE_DIR / "poi_category_counts.csv"
POI_CATEGORY_SHP = LANDUSE_DIR / "poi_categories_clipped.shp"

SEGMENT_DIR = OUTPUT_ROOT / "segments"
SEGMENT_MODEL_CSV_GZ = SEGMENT_DIR / "segment_model_table.csv.gz"
SEGMENT_MODEL_CSV = SEGMENT_DIR / "segment_model_table.csv"
VARIABLE_DICTIONARY_CSV = SEGMENT_DIR / "variable_dictionary.csv"

VIF_CSV = OUTPUT_ROOT / "diagnostics" / "vif_table.csv"
PATH_INFO_JSON = OUTPUT_ROOT / "effective_paths.json"


def first_existing_path(candidates, label, required=True):
    for p in candidates:
        p = Path(p)
        if p.exists():
            return p
    if required:
        raise FileNotFoundError("No existing %s found. Checked: %s" % (label, [str(p) for p in candidates]))
    return Path(candidates[0])


def find_optional_file(candidates, keyword_patterns=None):
    for p in candidates:
        p = Path(p)
        if p.exists():
            return p
    if keyword_patterns is None:
        return None
    exts = {".shp", ".gpkg", ".geojson", ".csv", ".xlsx"}
    found = []
    for p in BASE_DIR.rglob("*"):
        if not p.is_file() or p.suffix.lower() not in exts:
            continue
        name = p.name.lower()
        for pat in keyword_patterns:
            if pat.lower() in name:
                found.append(p)
                break
    return found[0] if found else None

SEGMENT_CSV = first_existing_path(SEGMENT_CSV_CANDIDATES, "segment csv", required=True)
GRAPHML_PATH = first_existing_path(GRAPHML_CANDIDATES, "graphml", required=True)
POI_INPUT = find_optional_file(POI_INPUT_CANDIDATES, keyword_patterns=["poi", "兴趣", "保定"])
REST_INPUT = find_optional_file(REST_INPUT_CANDIDATES, keyword_patterns=["restaurant", "餐馆", "餐饮", "大众点评"])

PATH_INFO = {
    "segment_csv": str(SEGMENT_CSV),
    "graphml_path": str(GRAPHML_PATH),
    "waybill_csv": str(WAYBILL_CSV),
    "wave_csv": str(WAVE_CSV),
    "poi_input": str(POI_INPUT) if POI_INPUT is not None else None,
    "restaurant_input": str(REST_INPUT) if REST_INPUT is not None else None,
    "output_root": str(OUTPUT_ROOT),
    "buffer_m": BUFFER_M,
}
PATH_INFO_JSON.parent.mkdir(parents=True, exist_ok=True)
PATH_INFO_JSON.write_text(json.dumps(PATH_INFO, indent=2, ensure_ascii=False), encoding="utf-8")
PATH_INFO


{'segment_csv': 'c:\\Users\\Kwk10\\Desktop\\2026 Spring\\Meituan\\outputs_new_clear\\OD\\MidPoints\\courier_final_distances_filtered.csv',
 'graphml_path': 'c:\\Users\\Kwk10\\Desktop\\2026 Spring\\Meituan\\baoding_clear.graphml',
 'waybill_csv': 'c:\\Users\\Kwk10\\Desktop\\2026 Spring\\Meituan\\all_waybill_info.csv',
 'wave_csv': 'c:\\Users\\Kwk10\\Desktop\\2026 Spring\\Meituan\\courier_wave_info.csv',
 'poi_input': 'c:\\Users\\Kwk10\\Desktop\\2026 Spring\\Meituan\\baoding\\保定数据\\POI_保定.shp',
 'restaurant_input': 'c:\\Users\\Kwk10\\Desktop\\2026 Spring\\Meituan\\baoding\\保定数据\\大众点评餐馆_保定.shp',
 'output_root': 'c:\\Users\\Kwk10\\Desktop\\2026 Spring\\Meituan\\outputs_new_clear\\segment_variables_clean',
 'buffer_m': 300}

## 2. Utilities

These helpers provide atomic writes, file reuse checks, path parsing, geometry helpers, and VIF support.


In [4]:
# =========================================================
# 2. Utilities
# =========================================================
def ensure_dir(path_like):
    Path(path_like).mkdir(parents=True, exist_ok=True)


def write_csv_atomic(df, path, index=False, compression=None):
    path = Path(path)
    ensure_dir(path.parent)
    tmp = path.with_suffix(path.suffix + ".tmp")
    df.to_csv(tmp, index=index, compression=compression)
    os.replace(str(tmp), str(path))
    print("[saved] %s rows=%s cols=%s" % (path, f"{len(df):,}", f"{df.shape[1]:,}"))


def save_json(obj, path):
    path = Path(path)
    ensure_dir(path.parent)
    tmp = path.with_suffix(path.suffix + ".tmp")
    tmp.write_text(json.dumps(obj, indent=2, ensure_ascii=False), encoding="utf-8")
    os.replace(str(tmp), str(path))


def required_columns_present(path, required_columns):
    path = Path(path)
    if not path.exists():
        return False
    if required_columns is None or len(required_columns) == 0:
        return True
    try:
        cols = set(pd.read_csv(path, nrows=0).columns)
    except Exception:
        return False
    missing = [c for c in required_columns if c not in cols]
    if missing:
        print("Existing file has missing columns:", path)
        print(missing[:30])
        return False
    return True


def should_reuse_file(path, force=False, required_columns=None):
    if force or not RESUME_FROM_EXISTING:
        return False
    return required_columns_present(path, required_columns)


def normalize_text(x):
    if pd.isna(x):
        return ""
    s = str(x).strip()
    if s in {"nan", "NaN", "<NA>", "None"}:
        return ""
    return s


def nonempty_value(x):
    s = normalize_text(x)
    if s == "":
        return None
    return s


def parse_boolish(x):
    s = normalize_text(x).lower()
    if s in {"true", "1", "yes"}:
        return True
    if s in {"false", "0", "no"}:
        return False
    return False


def add_time_columns(df, start_col="start_time"):
    out = df.copy()
    ts = pd.to_datetime(pd.to_numeric(out[start_col], errors="coerce"), unit="s", utc=True, errors="coerce")
    ts_local = ts + pd.Timedelta(hours=TZ_SHIFT_HOURS)
    out["start_ts_local"] = ts_local.dt.strftime("%Y-%m-%d %H:%M:%S")
    out["start_date_local"] = ts_local.dt.date.astype("string")
    out["start_hour_local"] = ts_local.dt.hour
    out["start_weekday_local"] = ts_local.dt.weekday
    out["is_weekend_local"] = (out["start_weekday_local"] >= 5).astype("float")
    out["is_workday_local"] = (out["start_weekday_local"] < 5).astype("float")
    out["start_hour_sin"] = np.sin(2.0 * np.pi * out["start_hour_local"].astype(float) / 24.0)
    out["start_hour_cos"] = np.cos(2.0 * np.pi * out["start_hour_local"].astype(float) / 24.0)
    return out


def parse_nodes(path_str):
    s = normalize_text(path_str)
    if s == "":
        return []
    try:
        return [int(x) for x in s.split(",") if str(x).strip() != ""]
    except Exception:
        return []


def parse_edge_id_str(s):
    s = normalize_text(s)
    if s == "":
        return None
    try:
        u, v, k = s.split("|")
        return int(u), int(v), int(k)
    except Exception:
        return None


def und_key(a, b):
    a = int(a)
    b = int(b)
    if a < b:
        return "%d|%d" % (a, b)
    return "%d|%d" % (b, a)


def parse_edge_key_list(s):
    s = normalize_text(s)
    if s == "":
        return []
    return [x for x in s.split(";") if x]


def safe_nanmean(vals):
    vals = np.asarray(vals, dtype=float)
    if vals.size == 0 or np.all(np.isnan(vals)):
        return np.nan
    return float(np.nanmean(vals))


def shannon_entropy(labels, global_K=None):
    labels = np.asarray(labels)
    labels = labels[pd.notna(labels)]
    if labels.size == 0:
        return 0.0, 0.0
    cnt = Counter(labels.tolist())
    total = float(sum(cnt.values()))
    if total <= 0:
        return 0.0, 0.0
    probs = np.asarray([v / total for v in cnt.values()], dtype=float)
    ent = -float(np.sum(probs * np.log(probs + 1e-12)))
    K = int(global_K) if global_K is not None and global_K > 1 else len(cnt)
    ent_norm = ent / (math.log(K) + 1e-12) if K > 1 else 0.0
    return ent, ent_norm


def line_angular_curvature_deg(geom):
    if geom is None:
        return 0.0
    if isinstance(geom, MultiLineString):
        return float(sum(line_angular_curvature_deg(g) for g in geom.geoms))
    if not isinstance(geom, LineString):
        return 0.0
    coords = np.asarray(geom.coords, dtype=float)
    if coords.shape[0] < 3:
        return 0.0
    dx = np.diff(coords[:, 0])
    dy = np.diff(coords[:, 1])
    ang = np.arctan2(dy, dx)
    d = np.diff(ang)
    d = (d + np.pi) % (2 * np.pi) - np.pi
    return float(np.sum(np.abs(np.degrees(d))))


def build_strtree(geoms):
    geoms = list(geoms)
    tree = STRtree(geoms)
    wkb_to_idx = {}
    for i, g in enumerate(geoms):
        try:
            wkb_to_idx[g.wkb] = i
        except Exception:
            pass
    return tree, geoms, wkb_to_idx


def strtree_query_indices(tree, geoms_list, wkb_to_idx, geom, predicate="intersects"):
    try:
        idxs = tree.query(geom, predicate=predicate)
        if len(idxs) and isinstance(idxs[0], (int, np.integer)):
            return [int(i) for i in idxs]
        if hasattr(idxs, "dtype") and np.issubdtype(idxs.dtype, np.integer):
            return [int(i) for i in idxs]
        out = []
        for g in idxs:
            j = wkb_to_idx.get(getattr(g, "wkb", None), None)
            if j is None:
                for k, gg in enumerate(geoms_list):
                    try:
                        if gg.equals(g):
                            j = k
                            break
                    except Exception:
                        pass
            if j is not None:
                out.append(int(j))
        return out
    except TypeError:
        idxs = tree.query(geom)
        out = []
        for g in idxs:
            try:
                if predicate == "intersects" and not g.intersects(geom):
                    continue
            except Exception:
                continue
            j = wkb_to_idx.get(getattr(g, "wkb", None), None)
            if j is not None:
                out.append(int(j))
        return out


def reorder_with_groups(df, grouped_specs, table_name):
    ordered = []
    seen = set()
    rows = []
    order_no = 0
    for role, group_name, cols in grouped_specs:
        for c in cols:
            if c in df.columns and c not in seen:
                ordered.append(c)
                seen.add(c)
                order_no += 1
                rows.append({"table": table_name, "column_name": c, "column_order": order_no, "role": role, "group_name": group_name, "in_default_model": int(role in {"x", "y", "weight"})})
    for c in df.columns:
        if c not in seen:
            ordered.append(c)
            order_no += 1
            rows.append({"table": table_name, "column_name": c, "column_order": order_no, "role": "other", "group_name": "unassigned", "in_default_model": 0})
    return df[ordered].copy(), pd.DataFrame(rows)


## 3. GraphML road class diagnostics

This section checks which edge attributes are available in `baoding_clear.graphml`. The road type variables in this notebook no longer assume OSM highway classes.


In [5]:
# =========================================================
# 3. GraphML diagnostics and road class mapping
# =========================================================
ROAD_CLASS_FIELD_PRIORITY = [
    "road_class_en", "EN_Class", "highway", "road_class_cn", "CN_Class",
]
ROAD_CLASS_CONTEXT_FIELDS = [
    "name", "ref", "shield", "parent_source_id", "Shape_Leng", "city", "district", "province",
]

ROAD_CLASS_CANONICAL_MAP = {
    "levelfourroad": "levelFourRoad",
    "levelthreeroad": "levelThreeRoad",
    "secondaryroad": "secondaryRoad",
    "nationalroad": "nationalRoad",
    "provincialroad": "provincialRoad",
    "overpass": "overPass",
    "四级道路": "levelFourRoad",
    "三级道路": "levelThreeRoad",
    "二级道路": "secondaryRoad",
    "国道": "nationalRoad",
    "国道其他路": "nationalRoadOther",
    "省道": "provincialRoad",
    "省道其他路": "provincialRoadOther",
    "天桥": "overPass",
}
ROAD_CLASS_BROAD_MAP = {
    "nationalRoad": "national_or_provincial",
    "nationalRoadOther": "national_or_provincial",
    "provincialRoad": "national_or_provincial",
    "provincialRoadOther": "national_or_provincial",
    "secondaryRoad": "secondary",
    "levelThreeRoad": "level_three",
    "levelFourRoad": "level_four",
    "overPass": "overpass",
    "unknown": "unknown",
}


def canonicalize_road_class(x):
    s = normalize_text(x)
    if s == "":
        return None
    # GraphML values may look like lists.
    if s.startswith("[") and s.endswith("]"):
        try:
            v = ast.literal_eval(s)
            if isinstance(v, list) and len(v) > 0:
                s = normalize_text(v[0])
        except Exception:
            pass
    key = re.sub(r"\s+", "", s).lower()
    if key in ROAD_CLASS_CANONICAL_MAP:
        return ROAD_CLASS_CANONICAL_MAP[key]
    # Preserve raw class if it is already a custom GraphML road class.
    return re.sub(r"[^0-9a-zA-Z_\u4e00-\u9fff]+", "_", s).strip("_") or None


def infer_road_class_from_attrs(attrs):
    for field in ROAD_CLASS_FIELD_PRIORITY:
        val = canonicalize_road_class(attrs.get(field, None))
        if val is not None:
            return val, field
    return "unknown", None


def broad_road_class(canonical):
    if canonical in ROAD_CLASS_BROAD_MAP:
        return ROAD_CLASS_BROAD_MAP[canonical]
    return "other"


def graphml_edge_attribute_diagnostics(graphml_path):
    G = nx.read_graphml(str(graphml_path))
    graph_info = {
        "graph_type": str(type(G)),
        "n_nodes": int(G.number_of_nodes()),
        "n_edges": int(G.number_of_edges()),
        "graph_attrs": dict(G.graph),
    }
    fields = ROAD_CLASS_FIELD_PRIORITY + ROAD_CLASS_CONTEXT_FIELDS + ["length", "geometry", "oneway", "reversed"]
    counters = {f: Counter() for f in fields}
    missing = {f: 0 for f in fields}
    class_counter = Counter()
    source_field_counter = Counter()
    iterator = G.edges(keys=True, data=True) if G.is_multigraph() else ((u, v, 0, d) for u, v, d in G.edges(data=True))
    for u, v, k, d in iterator:
        cls, src = infer_road_class_from_attrs(d)
        class_counter[cls] += 1
        source_field_counter[src or "missing"] += 1
        for f in fields:
            val = nonempty_value(d.get(f, None))
            if val is None:
                missing[f] += 1
            else:
                counters[f][str(val)] += 1
    diag = {"graph_info": graph_info, "field_missing_count": missing, "road_class_source_field_count": dict(source_field_counter)}
    diag["field_top_values"] = {f: counters[f].most_common(40) for f in fields}
    diag["canonical_road_class_count"] = dict(class_counter)
    save_json(diag, GRAPHML_DIAGNOSTIC_JSON)
    cls_df = pd.DataFrame([{"road_class": k, "edge_count_directed": v, "road_class_broad": broad_road_class(k)} for k, v in class_counter.items()]).sort_values("edge_count_directed", ascending=False)
    write_csv_atomic(cls_df, GRAPHML_ROAD_CLASS_CSV, index=False)
    print("GraphML diagnostics saved:", GRAPHML_DIAGNOSTIC_JSON)
    print("Road class values saved:", GRAPHML_ROAD_CLASS_CSV)
    return diag, cls_df

diag, road_class_values = graphml_edge_attribute_diagnostics(GRAPHML_PATH)
road_class_values.head(20)


[saved] c:\Users\Kwk10\Desktop\2026 Spring\Meituan\outputs_new_clear\segment_variables_clean\graph\graphml_road_class_values.csv rows=7 cols=3
GraphML diagnostics saved: c:\Users\Kwk10\Desktop\2026 Spring\Meituan\outputs_new_clear\segment_variables_clean\graph\graphml_attribute_diagnostics.json
Road class values saved: c:\Users\Kwk10\Desktop\2026 Spring\Meituan\outputs_new_clear\segment_variables_clean\graph\graphml_road_class_values.csv


,road_class,edge_count_directed,road_class_broad
0,unknown,115086,unknown
2,levelFourRoad,12928,level_four
4,levelThreeRoad,5270,level_three
5,secondaryRoad,2344,secondary
3,nationalRoad,708,national_or_provincial
1,provincialRoad,678,national_or_provincial
6,overPass,58,overpass


## 4. Segment path parsing

The new movement file stores one row per segment. The path is recovered from `segment_mode`, `path_backbone_segment`, `start_snap_edge`, and `end_snap_edge`. Older `path_node_ids` files are also supported.


In [ ]:
# =========================================================
# 4. Segment path parsing
# =========================================================
def detect_segment_schema(df):
    new_cols = {"segment_mode", "path_backbone_segment", "start_snap_edge", "end_snap_edge"}
    if new_cols.issubset(set(df.columns)):
        return "new_segment"
    if "path_node_ids" in df.columns:
        return "path_nodes"
    raise ValueError("Cannot detect path schema. Need new segment virtual columns or path_node_ids.")


def edge_keys_from_virtual_segment(row):
    edge_keys = []
    mode = normalize_text(row.get("segment_mode", ""))
    start_edge = parse_edge_id_str(row.get("start_snap_edge", ""))
    end_edge = parse_edge_id_str(row.get("end_snap_edge", ""))
    backbone_nodes = parse_nodes(row.get("path_backbone_segment", ""))
    if mode == "same_edge" and start_edge is not None and end_edge is not None:
        edge_keys.append(und_key(start_edge[0], start_edge[1]))
        return edge_keys
    start_label = None
    end_label = None
    if "->" in mode:
        parts = mode.split("->", 1)
        start_label = parts[0].strip()
        end_label = parts[1].strip()
    if start_edge is not None and (start_label in {"u", "v"} or mode == ""):
        edge_keys.append(und_key(start_edge[0], start_edge[1]))
    if len(backbone_nodes) >= 2:
        for u, v in zip(backbone_nodes[:-1], backbone_nodes[1:]):
            if int(u) != int(v):
                edge_keys.append(und_key(u, v))
    if end_edge is not None and (end_label in {"u", "v"} or mode == ""):
        edge_keys.append(und_key(end_edge[0], end_edge[1]))
    out = []
    for k in edge_keys:
        if len(out) == 0 or out[-1] != k:
            out.append(k)
    return out


def edge_keys_from_path_nodes(row):
    nodes = parse_nodes(row.get("path_node_ids", ""))
    if len(nodes) < 2:
        return []
    edge_keys = []
    for u, v in zip(nodes[:-1], nodes[1:]):
        if int(u) != int(v):
            edge_keys.append(und_key(u, v))
    out = []
    for k in edge_keys:
        if len(out) == 0 or out[-1] != k:
            out.append(k)
    return out


def build_segment_edge_keys(df):
    schema = detect_segment_schema(df)
    keys = []
    for _, row in tqdm(df.iterrows(), total=len(df), desc="parse segment paths"):
        if schema == "new_segment":
            k = edge_keys_from_virtual_segment(row)
            if len(k) == 0 and "path_node_ids" in df.columns:
                k = edge_keys_from_path_nodes(row)
        else:
            k = edge_keys_from_path_nodes(row)
        keys.append(";".join(k))
    return keys, schema


## 5. Rider event state and rider features

Rider features are built from the raw waybill and wave files. Wave ID is joined with date and courier ID because that combination uniquely identifies a wave in the data documentation. Rejected waybills with zero times are removed before reconstructing event state.


In [ ]:
# =========================================================
# 5. Raw event state and rider features
# =========================================================
def parse_order_ids(x):
    if pd.isna(x):
        return []
    s = str(x).strip()
    if s == "" or s.lower() in {"nan", "none", "<na>"}:
        return []
    s = s.replace("[", "").replace("]", "").replace("(", "").replace(")", "")
    out = []
    for part in re.split(r"[,;\s]+", s):
        if part.strip() == "":
            continue
        try:
            out.append(int(float(part)))
        except Exception:
            pass
    return out


def clean_raw_waybill(raw):
    df = raw.copy()
    df.columns = [str(c).strip() for c in df.columns]
    for c in ["order_id", "courier_id", "waybill_id"]:
        if c in df.columns:
            df[c] = pd.to_numeric(df[c], errors="coerce")
    if "is_courier_grabbed" in df.columns:
        grabbed_col = "is_courier_grabbed"
    elif "is_courier_grab" in df.columns:
        grabbed_col = "is_courier_grab"
    else:
        grabbed_col = None
    if grabbed_col is not None:
        df = df[pd.to_numeric(df[grabbed_col], errors="coerce") == 1].copy()
    if "grab_time" in df.columns:
        df = df[pd.to_numeric(df["grab_time"], errors="coerce") > 0].copy()
    for c in ["grab_time", "fetch_time", "arrive_time", "estimate_arrived_time"]:
        if c in df.columns:
            df[c] = pd.to_numeric(df[c], errors="coerce")
    return df


def clean_wave_orders(raw_wave):
    wave = raw_wave.copy()
    wave.columns = [str(c).strip() for c in wave.columns]
    rows = []
    for r in wave.itertuples(index=False):
        d = r._asdict()
        order_ids = parse_order_ids(d.get("order_ids"))
        for oid in order_ids:
            rows.append({"dt": d.get("dt"), "courier_id": d.get("courier_id"), "wave_id": d.get("wave_id"), "order_id": oid})
    out = pd.DataFrame(rows)
    for c in ["dt", "courier_id", "wave_id", "order_id"]:
        out[c] = pd.to_numeric(out[c], errors="coerce")
    return out.dropna(subset=["dt", "courier_id", "wave_id", "order_id"]).copy()


def build_raw_events(waybill, wave_orders):
    merged = pd.merge(wave_orders[["dt", "courier_id", "wave_id", "order_id"]], waybill, on=["courier_id", "order_id"], how="inner", suffixes=("_wave", ""))
    pieces = []
    specs = [("GRAB", "grab_time", 0), ("FETCH", "fetch_time", 1), ("DELIVER", "arrive_time", 2)]
    for action, tcol, rank in specs:
        if tcol not in merged.columns:
            continue
        tmp = merged[["dt", "courier_id", "wave_id", "order_id", "estimate_arrived_time", tcol]].copy()
        tmp[tcol] = pd.to_numeric(tmp[tcol], errors="coerce")
        tmp = tmp[tmp[tcol].fillna(0) > 0].copy()
        tmp = tmp.rename(columns={tcol: "time"})
        tmp["action"] = action
        tmp["action_rank"] = rank
        pieces.append(tmp)
    if len(pieces) == 0:
        return pd.DataFrame()
    events = pd.concat(pieces, ignore_index=True)
    events = events.sort_values(["dt", "courier_id", "wave_id", "time", "action_rank", "order_id"]).reset_index(drop=True)
    return events


def build_event_state(events):
    rows = []
    if len(events) == 0:
        return pd.DataFrame()
    deadline_lookup = events.drop_duplicates("order_id").set_index("order_id")["estimate_arrived_time"].to_dict()
    group_cols = ["dt", "courier_id", "wave_id"]
    for keys, g in tqdm(events.groupby(group_cols, sort=False), desc="event states"):
        active = set()
        max_onhand = 0
        event_index = 0
        for r in g.itertuples(index=False):
            oid = int(r.order_id)
            action = str(r.action)
            if action in {"GRAB", "FETCH"}:
                active.add(oid)
            elif action == "DELIVER":
                active.discard(oid)
            active_deadlines = []
            for a in active:
                d = deadline_lookup.get(a, np.nan)
                if pd.notna(d):
                    active_deadlines.append(float(d))
            min_deadline = min(active_deadlines) if active_deadlines else np.nan
            pressure_sec = min_deadline - float(r.time) if pd.notna(min_deadline) else np.nan
            max_onhand = max(max_onhand, len(active))
            rows.append({
                "dt": int(r.dt), "courier_id": int(r.courier_id), "wave_id": int(r.wave_id),
                "event_index_in_wave": int(event_index), "event_time": int(r.time),
                "event_action": action, "event_order_id": oid,
                "onhand_order_count_after_event": int(len(active)),
                "onhand_order_ids_after_event": ",".join(str(x) for x in sorted(active)),
                "min_deadline_time_after_event": min_deadline,
                "time_pressure_sec_after_event": pressure_sec,
                "max_onhand_so_far_in_wave": int(max_onhand),
            })
            event_index += 1
    return pd.DataFrame(rows)


def get_or_build_event_state():
    req = ["dt", "courier_id", "wave_id", "event_time", "event_action", "event_order_id", "onhand_order_count_after_event"]
    if should_reuse_file(RAW_EVENT_STATE_CSV, force=FORCE_REBUILD_RAW_EVENTS, required_columns=req):
        print("[reuse] raw event state")
        return pd.read_csv(RAW_EVENT_STATE_CSV)
    raw_waybill = pd.read_csv(WAYBILL_CSV)
    raw_wave = pd.read_csv(WAVE_CSV)
    waybill = clean_raw_waybill(raw_waybill)
    wave_orders = clean_wave_orders(raw_wave)
    events = build_raw_events(waybill, wave_orders)
    state = build_event_state(events)
    write_csv_atomic(state, RAW_EVENT_STATE_CSV, index=False, compression="gzip")
    return state


def build_rider_features(event_state, segment_df):
    ev = event_state.copy()
    for c in ["dt", "courier_id", "wave_id", "event_time", "event_order_id"]:
        if c in ev.columns:
            ev[c] = pd.to_numeric(ev[c], errors="coerce")
    ev["event_date"] = ev["dt"].astype("Int64").astype("string")
    ev["event_date_ts"] = pd.to_datetime(ev["event_date"], format="%Y%m%d", errors="coerce")
    ev["event_weekday"] = ev["event_date_ts"].dt.weekday
    grab_ev = ev[ev["event_action"] == "GRAB"].copy()
    active_days = ev.groupby("courier_id")["dt"].nunique().rename("rider_active_days")
    active_weekdays = ev.groupby("courier_id")["event_weekday"].nunique().rename("rider_active_weekday_count")
    total_data_days = max(1, int(ev["dt"].nunique()))
    order_cnt = grab_ev.groupby("courier_id")["event_order_id"].nunique().rename("rider_total_orders_raw")
    avg_orders_day = (order_cnt / active_days).rename("rider_avg_orders_per_active_day")
    state_agg = ev.groupby("courier_id").agg(
        rider_mean_onhand_raw=("onhand_order_count_after_event", "mean"),
        rider_median_onhand_raw=("onhand_order_count_after_event", "median"),
        rider_share_event_onhand_ge2_raw=("onhand_order_count_after_event", lambda s: float(np.mean(pd.to_numeric(s, errors="coerce") >= 2))),
        rider_share_event_onhand_ge3_raw=("onhand_order_count_after_event", lambda s: float(np.mean(pd.to_numeric(s, errors="coerce") >= 3))),
    )
    wave_max = ev.groupby(["courier_id", "dt", "wave_id"])["onhand_order_count_after_event"].max().reset_index()
    wave_agg = wave_max.groupby("courier_id").agg(
        rider_mean_max_onhand_per_wave=("onhand_order_count_after_event", "mean"),
        rider_median_max_onhand_per_wave=("onhand_order_count_after_event", "median"),
        rider_share_single_order_wave=("onhand_order_count_after_event", lambda s: float(np.mean(pd.to_numeric(s, errors="coerce") <= 1))),
        rider_share_batch_wave_ge2=("onhand_order_count_after_event", lambda s: float(np.mean(pd.to_numeric(s, errors="coerce") >= 2))),
        rider_share_batch_wave_ge3=("onhand_order_count_after_event", lambda s: float(np.mean(pd.to_numeric(s, errors="coerce") >= 3))),
    )
    grab_ev["grab_when_already_onhand"] = (pd.to_numeric(grab_ev["onhand_order_count_after_event"], errors="coerce") >= 2).astype(float)
    grab_pattern = grab_ev.groupby("courier_id").agg(rider_share_grab_when_already_onhand=("grab_when_already_onhand", "mean"))
    seg = segment_df.copy()
    seg["final_distance_m"] = pd.to_numeric(seg.get("final_distance_m", np.nan), errors="coerce")
    dist_agg = seg.groupby("courier_id").agg(
        rider_mean_segment_distance_m=("final_distance_m", "mean"),
        rider_median_segment_distance_m=("final_distance_m", "median"),
        rider_share_long_segment=("final_distance_m", lambda s: float(np.mean(pd.to_numeric(s, errors="coerce") >= LONG_SEGMENT_THRESHOLD_M))),
    )
    out = pd.concat([active_days, active_weekdays, order_cnt, avg_orders_day, state_agg, wave_agg, grab_pattern, dist_agg], axis=1).reset_index()
    out["rider_active_day_share_in_data"] = out["rider_active_days"] / float(total_data_days)
    out["rider_works_all_7_weekdays"] = (out["rider_active_weekday_count"] >= 7).astype(int)
    out["rider_full_time_like"] = ((out["rider_active_day_share_in_data"] >= 0.80) | (out["rider_works_all_7_weekdays"] == 1)).astype(int)
    return out


def get_or_build_rider_features(event_state, segment_df):
    req = ["courier_id", "rider_avg_orders_per_active_day", "rider_full_time_like", "rider_mean_max_onhand_per_wave"]
    if should_reuse_file(RIDER_FEATURE_CSV, force=FORCE_REBUILD_RIDER_FEATURES, required_columns=req):
        print("[reuse] rider features")
        return pd.read_csv(RIDER_FEATURE_CSV)
    rider = build_rider_features(event_state, segment_df)
    write_csv_atomic(rider, RIDER_FEATURE_CSV, index=False)
    return rider


def attach_event_state_to_segments(segment_df, event_state):
    seg = segment_df.copy()
    ev = event_state.copy()
    key_left = ["dt", "courier_id", "wave_id", "start_time", "start_action", "start_order_id"]
    key_right = ["dt", "courier_id", "wave_id", "event_time", "event_action", "event_order_id"]
    for c in ["dt", "courier_id", "wave_id", "start_time", "start_order_id"]:
        if c in seg.columns:
            seg[c] = pd.to_numeric(seg[c], errors="coerce")
    for c in ["dt", "courier_id", "wave_id", "event_time", "event_order_id"]:
        if c in ev.columns:
            ev[c] = pd.to_numeric(ev[c], errors="coerce")
    ev_small = ev[key_right + ["onhand_order_count_after_event", "min_deadline_time_after_event", "time_pressure_sec_after_event", "max_onhand_so_far_in_wave"]].copy()
    ev_small = ev_small.drop_duplicates(key_right, keep="first")
    merged = seg.merge(ev_small, left_on=key_left, right_on=key_right, how="left")
    merged = merged.drop(columns=[c for c in key_right if c in merged.columns and c not in {"dt", "courier_id", "wave_id"}], errors="ignore")
    merged = merged.rename(columns={"onhand_order_count_after_event": "onhand_order_count_start", "max_onhand_so_far_in_wave": "max_onhand_so_far_in_wave_start"})
    if "time_pressure_sec" not in merged.columns or merged["time_pressure_sec"].isna().all():
        merged["time_pressure_sec"] = merged["time_pressure_sec_after_event"]
        merged["time_pressure_min"] = merged["time_pressure_sec"] / 60.0
    merged = merged.drop(columns=["time_pressure_sec_after_event"], errors="ignore")
    return merged


## 6. Graph core, road class refresh, and progress centrality

The expensive graph topology and centrality components are cached separately from road class shares. If previous centrality outputs exist, the road class correction can update only the road class columns.


In [ ]:
# =========================================================
# 6. Graph core, road class refresh, and progress centrality
# =========================================================
def load_graphml_as_undirected_edges(graphml_path):
    G = nx.read_graphml(str(graphml_path))
    crs = G.graph.get("crs", "EPSG:32650")
    node_rows = []
    node_geom = {}
    for n, d in G.nodes(data=True):
        try:
            ni = int(n)
            x = float(d.get("x"))
            y = float(d.get("y"))
        except Exception:
            continue
        geom = Point(x, y)
        node_geom[ni] = geom
        node_rows.append({"node": ni, "x": x, "y": y, "geometry": geom})
    gdf_nodes = gpd.GeoDataFrame(node_rows, geometry="geometry", crs=crs)
    edge_by_key = {}
    iterator = G.edges(keys=True, data=True) if G.is_multigraph() else ((u, v, 0, d) for u, v, d in G.edges(data=True))
    for u, v, k, d in tqdm(iterator, total=G.number_of_edges(), desc="read graphml edges"):
        try:
            ui = int(u)
            vi = int(v)
        except Exception:
            continue
        key = und_key(ui, vi)
        geom = d.get("geometry", None)
        if isinstance(geom, str) and len(geom) > 0:
            try:
                geom = wkt.loads(geom)
            except Exception:
                geom = None
        if geom is None:
            pu = node_geom.get(ui)
            pv = node_geom.get(vi)
            if pu is None or pv is None:
                continue
            geom = LineString([pu, pv])
        try:
            length = float(d.get("length", geom.length))
        except Exception:
            length = float(geom.length)
        cls, cls_src = infer_road_class_from_attrs(d)
        rec = {
            "u": ui, "v": vi,
            "u2": min(ui, vi), "v2": max(ui, vi),
            "und_key": key,
            "length_m": length,
            "geometry": geom,
            "road_class": cls,
            "road_class_source_field": cls_src or "missing",
            "road_class_broad": broad_road_class(cls),
            "name": nonempty_value(d.get("name", None)),
            "ref": nonempty_value(d.get("ref", d.get("shield", None))),
            "oneway": parse_boolish(d.get("oneway", False)),
            "reversed": parse_boolish(d.get("reversed", False)),
        }
        # Prefer non-missing road class; otherwise keep shortest geometry.
        prev = edge_by_key.get(key)
        if prev is None:
            edge_by_key[key] = rec
        else:
            prev_missing = prev["road_class"] == "unknown"
            cur_missing = rec["road_class"] == "unknown"
            replace = False
            if prev_missing and not cur_missing:
                replace = True
            elif prev_missing == cur_missing and rec["length_m"] < prev["length_m"]:
                replace = True
            if replace:
                edge_by_key[key] = rec
    gdf_und = gpd.GeoDataFrame(list(edge_by_key.values()), geometry="geometry", crs=crs).reset_index(drop=True)
    gdf_und["crowfly_m"] = [node_geom[int(u)].distance(node_geom[int(v)]) if int(u) in node_geom and int(v) in node_geom else np.nan for u, v in zip(gdf_und["u2"], gdf_und["v2"])]
    gdf_und["tortuosity_edge"] = gdf_und["length_m"] / gdf_und["crowfly_m"].clip(lower=1e-6)
    gdf_und["curvature_deg_edge"] = [line_angular_curvature_deg(g) for g in tqdm(gdf_und.geometry.values, desc="edge curvature")]
    gdf_und["curvature_deg_per_m_edge"] = gdf_und["curvature_deg_edge"] / gdf_und["length_m"].clip(lower=1e-6)
    return G, gdf_nodes, gdf_und


def build_undirected_graph(gdf_und):
    G = nx.Graph()
    for r in gdf_und.itertuples(index=False):
        a = int(r.u2)
        b = int(r.v2)
        if a == b:
            continue
        G.add_edge(a, b, length=float(r.length_m), und_key=str(r.und_key))
    return G


def contract_degree2(G):
    conedges = []
    und_to_con = {}
    terminals = set([n for n in G.nodes if G.degree(n) != 2])
    if len(terminals) == 0 and G.number_of_nodes() > 0:
        terminals.add(next(iter(G.nodes)))
    visited = set()
    def ekey(a, b):
        return (a, b) if a < b else (b, a)
    con_id = 0
    for t in tqdm(list(terminals), desc="contract degree two graph"):
        for nbr in list(G.neighbors(t)):
            e = ekey(t, nbr)
            if e in visited:
                continue
            path_edges = [e]
            visited.add(e)
            total_len = float(G.edges[e].get("length", 0.0))
            prev = t
            curr = nbr
            while curr not in terminals:
                nbrs = list(G.neighbors(curr))
                if len(nbrs) != 2:
                    break
                nxt = nbrs[0] if nbrs[1] == prev else nbrs[1]
                e2 = ekey(curr, nxt)
                if e2 in visited:
                    break
                visited.add(e2)
                path_edges.append(e2)
                total_len += float(G.edges[e2].get("length", 0.0))
                prev, curr = curr, nxt
            con_id += 1
            conedges.append({"conedge_id": int(con_id), "u": int(t), "v": int(curr), "length_m": float(total_len), "orig_edges": path_edges})
            for a, b in path_edges:
                und_to_con[und_key(a, b)] = int(con_id)
    Gc = nx.Graph()
    for ce in conedges:
        Gc.add_edge(int(ce["u"]), int(ce["v"]), length=float(ce["length_m"]), conedge_id=int(ce["conedge_id"]))
    return conedges, Gc, und_to_con


def approx_betweenness_with_progress(G, k=300, seed=42, weight="length", batch_size=20):
    nodes = list(G.nodes())
    if len(nodes) == 0 or k <= 0:
        return {}
    rng = np.random.default_rng(seed)
    k = min(int(k), len(nodes))
    sources = rng.choice(np.asarray(nodes), size=k, replace=False).tolist()
    acc = dict((n, 0.0) for n in nodes)
    batches = [sources[i:i + batch_size] for i in range(0, len(sources), batch_size)]
    for batch in tqdm(batches, desc="approx betweenness batches"):
        b = nx.betweenness_centrality_subset(G, sources=batch, targets=nodes, normalized=False, weight=weight)
        for n, val in b.items():
            acc[n] += float(val)
    scale = float(len(nodes)) / float(k)
    for n in acc:
        acc[n] *= scale
    return acc


def approx_farness_with_progress(G, n_sources=200, seed=42):
    nodes = list(G.nodes())
    if len(nodes) == 0:
        return {}
    rng = np.random.default_rng(seed)
    sample_n = min(int(n_sources), len(nodes))
    sources = rng.choice(np.asarray(nodes), size=sample_n, replace=False)
    acc = defaultdict(float)
    cnt = defaultdict(int)
    for s in tqdm(sources, desc="approx farness sources"):
        lengths = nx.single_source_dijkstra_path_length(G, int(s), weight="length")
        for n, d in lengths.items():
            acc[int(n)] += float(d)
            cnt[int(n)] += 1
    out = {}
    for n in nodes:
        c = cnt.get(int(n), 0)
        out[int(n)] = acc.get(int(n), np.nan) / c if c > 0 else np.nan
    return out


def conedge_geometry_from_edges(path_edges, und_geom):
    geoms = []
    for a, b in path_edges:
        key = und_key(a, b)
        g = und_geom.get(key)
        if g is not None:
            geoms.append(g)
    if len(geoms) == 0:
        return None
    try:
        return linemerge(unary_union(geoms))
    except Exception:
        return unary_union(geoms)


def build_graph_core():
    raw_graph, gdf_nodes, gdf_und = load_graphml_as_undirected_edges(GRAPHML_PATH)
    G_und = build_undirected_graph(gdf_und)
    conedges, Gc, und_to_con = contract_degree2(G_und)
    print("Contracted graph nodes=%s conedges=%s" % (f"{Gc.number_of_nodes():,}", f"{len(conedges):,}"))
    und_len = dict(zip(gdf_und["und_key"].astype(str), gdf_und["length_m"].astype(float)))
    und_geom = dict(zip(gdf_und["und_key"].astype(str), gdf_und.geometry.values))
    node_geom = gdf_nodes.set_index("node").geometry.to_dict()
    node_degree = dict(Gc.degree())
    if COMPUTE_BETWEENNESS_IF_MISSING:
        k = min(int(BETWEENNESS_K), Gc.number_of_nodes())
        print("Computing approximate betweenness with progress. sampled sources =", k)
        node_btw = approx_betweenness_with_progress(Gc, k=k, seed=RANDOM_SEED, weight="length", batch_size=BETWEENNESS_BATCH_SIZE)
    else:
        node_btw = {}
    if COMPUTE_CLOSENESS_IF_MISSING:
        print("Computing approximate closeness/farness with progress. sampled sources =", FARNESS_SAMPLE_SOURCES)
        far = approx_farness_with_progress(Gc, n_sources=FARNESS_SAMPLE_SOURCES, seed=RANDOM_SEED)
        node_close = {}
        for n, v in far.items():
            node_close[int(n)] = 1.0 / float(v) if pd.notna(v) and float(v) > 0 else np.nan
    else:
        node_close = {}
    rows = []
    for ce in tqdm(conedges, desc="conedge core rows"):
        cid = int(ce["conedge_id"])
        u = int(ce["u"])
        v = int(ce["v"])
        L = float(ce["length_m"])
        geom = conedge_geometry_from_edges(ce["orig_edges"], und_geom)
        p_u = node_geom.get(u)
        p_v = node_geom.get(v)
        crow = float(p_u.distance(p_v)) if p_u is not None and p_v is not None else np.nan
        tort = float(L / max(crow, 1e-6)) if pd.notna(crow) else np.nan
        lac = line_angular_curvature_deg(geom)
        rows.append({
            "conedge_id": cid, "u": u, "v": v, "length_m": L, "crowfly_m": crow,
            "tortuosity": tort,
            "curvature_deg": lac,
            "curvature_deg_per_m": lac / max(L, 1e-6),
            "degree_mean_end": safe_nanmean([node_degree.get(u, np.nan), node_degree.get(v, np.nan)]),
            "betweenness_raw_mean_end": safe_nanmean([node_btw.get(u, np.nan), node_btw.get(v, np.nan)]),
            "closeness_approx_mean_end": safe_nanmean([node_close.get(u, np.nan), node_close.get(v, np.nan)]),
            "geometry": geom,
            "orig_edge_keys": ";".join([und_key(a, b) for a, b in ce["orig_edges"]]),
        })
    gdf_con = gpd.GeoDataFrame(rows, geometry="geometry", crs=gdf_und.crs)
    add_local_network_features(gdf_con, Gc, conedges)
    cache = {"gdf_nodes": gdf_nodes, "gdf_und": gdf_und, "gdf_con": gdf_con, "und_to_con": und_to_con, "und_len": und_len, "und_geom": und_geom, "graph_crs": str(gdf_und.crs)}
    return cache


def add_local_network_features(gdf_con, Gc, conedges):
    if "road_len_in_300m" in gdf_con.columns and "intersection_density_per_km_300m" in gdf_con.columns:
        return gdf_con
    print("Computing network radius features with progress. radius=%dm" % BUFFER_M)
    node_cutoff = {}
    for n in tqdm(Gc.nodes, desc="node Dijkstra cutoff"):
        node_cutoff[int(n)] = nx.single_source_dijkstra_path_length(Gc, int(n), cutoff=BUFFER_M, weight="length")
    conedge_endpoints = {}
    conedge_len = {}
    incident = defaultdict(set)
    for ce in conedges:
        cid = int(ce["conedge_id"])
        u = int(ce["u"])
        v = int(ce["v"])
        L = float(ce["length_m"])
        conedge_endpoints[cid] = (u, v)
        conedge_len[cid] = L
        incident[u].add(cid)
        incident[v].add(cid)
    node_degree = {int(n): int(d) for n, d in Gc.degree()}
    jnodes = set([int(n) for n, d in Gc.degree() if int(d) >= 2])
    len_vals = []
    jnc_vals = []
    conn_vals = []
    int_density_vals = []
    for cid in tqdm(gdf_con["conedge_id"].astype(int).tolist(), desc="conedge local 300m"):
        u, v = conedge_endpoints[cid]
        du = node_cutoff.get(u, {})
        dv = node_cutoff.get(v, {})
        nodes = set(du.keys()) | set(dv.keys())
        j = [n for n in nodes if int(n) in jnodes]
        cand = set()
        for n in nodes:
            cand |= incident.get(int(n), set())
        local_len = 0.0
        for cid2 in cand:
            a, b = conedge_endpoints[cid2]
            L = conedge_len[cid2]
            da = min(float(du.get(a, np.inf)), float(dv.get(a, np.inf)))
            db = min(float(du.get(b, np.inf)), float(dv.get(b, np.inf)))
            local_len += min(float(L), max(0.0, BUFFER_M - da) + max(0.0, BUFFER_M - db))
        jnc = len(j)
        conn = sum(node_degree.get(int(n), 0) for n in j)
        len_vals.append(float(local_len))
        jnc_vals.append(float(jnc))
        conn_vals.append(float(conn))
        int_density_vals.append(float(jnc) / max(local_len / 1000.0, 1e-6))
    gdf_con["road_len_in_300m"] = len_vals
    gdf_con["intersection_count_in_300m"] = jnc_vals
    gdf_con["connectivity_sum_degree_in_300m"] = conn_vals
    gdf_con["intersection_density_per_km_300m"] = int_density_vals
    return gdf_con


def load_existing_graph_cache():
    for p in [GRAPH_CORE_CACHE_PKL, GRAPH_FEATURE_PKL]:
        if p.exists():
            try:
                with open(p, "rb") as f:
                    cache = pickle.load(f)
                if "gdf_con" in cache and "und_to_con" in cache and "und_len" in cache:
                    print("[reuse] graph cache:", p)
                    return cache
            except Exception as exc:
                print("Could not load graph cache", p, exc)
    return None


def save_graph_cache(cache):
    ensure_dir(GRAPH_CORE_CACHE_PKL.parent)
    with open(GRAPH_CORE_CACHE_PKL, "wb") as f:
        pickle.dump(cache, f)
    with open(GRAPH_FEATURE_PKL, "wb") as f:
        pickle.dump(cache, f)
    print("[saved] graph cache:", GRAPH_CORE_CACHE_PKL)


def refresh_conedge_road_classes(cache):
    print("Refreshing road class columns from GraphML without recomputing centrality")
    _, _, gdf_und_current = load_graphml_as_undirected_edges(GRAPHML_PATH)
    class_by_key = {}
    broad_by_key = {}
    len_by_key = {}
    for r in gdf_und_current.itertuples(index=False):
        key = str(r.und_key)
        class_by_key[key] = str(r.road_class)
        broad_by_key[key] = str(r.road_class_broad)
        len_by_key[key] = float(r.length_m)
    gdf_con = cache["gdf_con"].copy()
    for c in list(gdf_con.columns):
        if c.startswith("road_class_share_") or c.startswith("road_class_broad_share_"):
            gdf_con = gdf_con.drop(columns=[c])
    observed_classes = sorted(set(class_by_key.values()) | {"unknown"})
    observed_broad = sorted(set(broad_by_key.values()) | {"unknown"})
    rows = []
    for r in tqdm(gdf_con.itertuples(index=False), total=len(gdf_con), desc="conedge road class shares"):
        cid = int(getattr(r, "conedge_id"))
        edge_keys = parse_edge_key_list(getattr(r, "orig_edge_keys", ""))
        if len(edge_keys) == 0:
            # Backward compatibility when cache does not have orig_edge_keys.
            edge_keys = [k for k, c in cache["und_to_con"].items() if int(c) == cid]
        class_w = Counter()
        broad_w = Counter()
        total = 0.0
        for key in edge_keys:
            w = float(len_by_key.get(key, cache.get("und_len", {}).get(key, 0.0)))
            if w <= 0:
                continue
            cls = class_by_key.get(key, "unknown")
            br = broad_by_key.get(key, broad_road_class(cls))
            class_w[cls] += w
            broad_w[br] += w
            total += w
        rec = {"conedge_id": cid, "road_class_major": "unknown", "road_class_broad_major": "unknown"}
        if total > 0:
            rec["road_class_major"] = class_w.most_common(1)[0][0]
            rec["road_class_broad_major"] = broad_w.most_common(1)[0][0]
        for cls in observed_classes:
            rec["road_class_share_%s" % cls] = float(class_w.get(cls, 0.0) / total) if total > 0 else np.nan
        for br in observed_broad:
            rec["road_class_broad_share_%s" % br] = float(broad_w.get(br, 0.0) / total) if total > 0 else np.nan
        rows.append(rec)
    class_df = pd.DataFrame(rows)
    gdf_con = gdf_con.merge(class_df, on="conedge_id", how="left")
    cache["gdf_con"] = gdf_con
    cache["gdf_und"] = gdf_und_current
    cache["und_len"] = dict(zip(gdf_und_current["und_key"].astype(str), gdf_und_current["length_m"].astype(float)))
    cache["und_geom"] = dict(zip(gdf_und_current["und_key"].astype(str), gdf_und_current.geometry.values))
    return cache


def get_or_build_graph_features():
    cache = None
    if RESUME_FROM_EXISTING and not FORCE_REBUILD_GRAPH_CORE:
        cache = load_existing_graph_cache()
    if cache is None:
        cache = build_graph_core()
    if FORCE_REFRESH_ROAD_CLASS or "road_class_share_levelFourRoad" not in cache["gdf_con"].columns:
        cache = refresh_conedge_road_classes(cache)
    save_graph_cache(cache)
    gdf_con = cache["gdf_con"]
    write_csv_atomic(gdf_con.drop(columns=["geometry"], errors="ignore"), CONEDGE_FEATURE_CSV, index=False, compression="gzip")
    if WRITE_GPKG:
        try:
            gdf_con.to_file(CONEDGE_GPKG, layer="conedge_features", driver="GPKG")
            print("[saved]", CONEDGE_GPKG)
        except Exception as exc:
            print("Warning: cannot write conedge gpkg:", exc)
    return cache


## 7. POI and land use features

POI input can be a shapefile, GeoPackage, GeoJSON, CSV, or Excel table. CSV input needs either a WKT geometry column or coordinate columns such as `lng` and `lat`.


In [ ]:
# =========================================================
# 7. POI and land use features
# =========================================================
POI_TARGET_KEYWORDS = OrderedDict([
    # Restaurant is mainly handled by REST_SHP when available, but keep keywords for POI tables that contain restaurants.
    ("restaurant", ["餐饮", "美食", "餐馆", "饭店", "restaurant", "food"]),
    # Categories observed in poi_category_counts.csv.
    ("traffic_facility", ["交通设施服务"]),
    ("passage_facility", ["通行设施"]),
    ("residential_business", ["商务住宅", "小区", "住宅", "社区", "residential", "community"]),
    ("company", ["公司企业"]),
    ("government", ["政府机构及社会团体"]),
    ("finance", ["金融保险服务"]),
    ("education_culture", ["科教文化服务"]),
    ("accommodation", ["住宿服务"]),
    ("public_facility", ["公共设施"]),
    ("scenic", ["风景名胜"]),
    ("shopping", ["购物服务"]),
    ("life_service", ["生活服务"]),
])


def infer_poi_type_column(gdf):
    candidates = ["type", "category", "cat", "poi_type", "class", "subclass", "一级分类", "二级分类", "类别", "分类", "大类", "小类", "NAME", "name", "kind", "Kind"]
    cols = set([str(c) for c in gdf.columns])
    for c in candidates:
        if c in cols:
            return c
    obj_cols = [c for c in gdf.columns if c != gdf.geometry.name and pd.api.types.is_object_dtype(gdf[c])]
    best = None
    best_score = -1
    for c in obj_cols:
        nun = gdf[c].nunique(dropna=True)
        if nun > 1 and nun < max(500, len(gdf) * 0.5):
            score = len(gdf) - nun
            if score > best_score:
                best = c
                best_score = score
    return best


def extract_major_type(series):
    s = series.fillna("unknown").astype(str)
    s = s.str.replace("；", ";", regex=False)
    s = s.str.replace("|", ";", regex=False)
    major = s.str.split(";", n=1).str[0].str.strip()
    major = major.replace({"": "unknown", "nan": "unknown", "None": "unknown", "<NA>": "unknown"})
    return major


def match_keywords(series, keywords):
    s = series.fillna("").astype(str).str.lower()
    mask = pd.Series(False, index=s.index)
    for kw in keywords:
        mask = mask | s.str.contains(str(kw).lower(), regex=False)
    return mask


def infer_lon_lat_columns(df):
    lon_candidates = ["lng", "lon", "longitude", "x", "经度", "X"]
    lat_candidates = ["lat", "latitude", "y", "纬度", "Y"]
    lon_col = None
    lat_col = None
    for c in lon_candidates:
        if c in df.columns:
            lon_col = c
            break
    for c in lat_candidates:
        if c in df.columns:
            lat_col = c
            break
    return lon_col, lat_col


def load_any_spatial_table(path, target_crs):
    if path is None or not Path(path).exists():
        return None
    path = Path(path)
    suffix = path.suffix.lower()
    if suffix in {".shp", ".gpkg", ".geojson"}:
        gdf = gpd.read_file(path)
    elif suffix in {".csv", ".txt"}:
        df = pd.read_csv(path)
        geom_col = None
        for c in ["geometry", "geom", "wkt", "WKT"]:
            if c in df.columns:
                geom_col = c
                break
        if geom_col is not None:
            geom = df[geom_col].apply(lambda x: wkt.loads(x) if pd.notna(x) and str(x).strip() != "" else None)
            gdf = gpd.GeoDataFrame(df, geometry=geom, crs="EPSG:4326")
        else:
            lon_col, lat_col = infer_lon_lat_columns(df)
            if lon_col is None or lat_col is None:
                raise ValueError("CSV POI file needs WKT geometry or lon/lat columns: %s" % path)
            geom = [Point(xy) for xy in zip(pd.to_numeric(df[lon_col], errors="coerce"), pd.to_numeric(df[lat_col], errors="coerce"))]
            gdf = gpd.GeoDataFrame(df, geometry=geom, crs="EPSG:4326")
    elif suffix in {".xlsx", ".xls"}:
        df = pd.read_excel(path)
        lon_col, lat_col = infer_lon_lat_columns(df)
        if lon_col is None or lat_col is None:
            raise ValueError("Excel POI file needs lon/lat columns: %s" % path)
        geom = [Point(xy) for xy in zip(pd.to_numeric(df[lon_col], errors="coerce"), pd.to_numeric(df[lat_col], errors="coerce"))]
        gdf = gpd.GeoDataFrame(df, geometry=geom, crs="EPSG:4326")
    else:
        raise ValueError("Unsupported spatial table type: %s" % path)
    if gdf.crs is None:
        gdf = gdf.set_crs("EPSG:4326")
    gdf = gdf[gdf.geometry.notnull() & ~gdf.geometry.is_empty].copy()
    return gdf.to_crs(target_crs)


def build_conedge_landuse(cache):
    gdf_con = cache["gdf_con"].copy()
    target_crs = gdf_con.crs
    if POI_INPUT is None:
        print("POI input not found. Land use features will be empty.")
        return pd.DataFrame({"conedge_id": gdf_con["conedge_id"].values})
    print("Loading POI input:", POI_INPUT)
    poi = load_any_spatial_table(POI_INPUT, target_crs)
    if poi is None or len(poi) == 0:
        return pd.DataFrame({"conedge_id": gdf_con["conedge_id"].values})
    minx, miny, maxx, maxy = gdf_con.total_bounds
    bbox_geom = box(minx - BUFFER_M - 50, miny - BUFFER_M - 50, maxx + BUFFER_M + 50, maxy + BUFFER_M + 50)
    poi = poi[poi.geometry.intersects(bbox_geom)].copy()
    poi_type_col = infer_poi_type_column(poi)
    if poi_type_col is None:
        poi["poi_major"] = "unknown"
        poi_type_col = "poi_major"
    else:
        poi["poi_major"] = extract_major_type(poi[poi_type_col])
    poi["poi_type_source_col"] = poi_type_col
    category_counts = poi["poi_major"].value_counts(dropna=False).reset_index()
    category_counts.columns = ["poi_major", "count"]
    write_csv_atomic(category_counts, POI_CATEGORY_CSV, index=False)
    if WRITE_SHP:
        try:
            ensure_dir(POI_CATEGORY_SHP.parent)
            poi_export = poi[["poi_major", "poi_type_source_col", "geometry"]].copy()
            poi_export["poi_major"] = poi_export["poi_major"].astype(str).str.slice(0, 80)
            poi_export.to_file(POI_CATEGORY_SHP, encoding="utf-8")
            print("[saved]", POI_CATEGORY_SHP)
        except Exception as exc:
            print("Warning: cannot write POI category shapefile:", exc)
    rest = load_any_spatial_table(REST_INPUT, target_crs) if REST_INPUT is not None else None
    poi_tree, poi_geoms, poi_wkb = build_strtree(list(poi.geometry.values))
    poi_major = poi["poi_major"].values
    global_K = max(1, int(pd.Series(poi_major).nunique(dropna=True)))
    if rest is not None and len(rest) > 0:
        rest = rest[rest.geometry.intersects(bbox_geom)].copy()
        rest_tree, rest_geoms, rest_wkb = build_strtree(list(rest.geometry.values))
    else:
        rest_tree, rest_geoms, rest_wkb = None, [], {}
    target_masks = {}
    for name, kws in POI_TARGET_KEYWORDS.items():
        target_masks[name] = match_keywords(poi["poi_major"], kws).values
    rows = []
    for r in tqdm(gdf_con.itertuples(index=False), total=len(gdf_con), desc="landuse 300m buffers"):
        geom = getattr(r, "geometry")
        cid = int(getattr(r, "conedge_id"))
        if geom is None or geom.is_empty:
            rec = {"conedge_id": cid, "poi_count_300m": 0, "poi_mix_entropy_norm_300m": 0.0}
            for name in POI_TARGET_KEYWORDS.keys():
                rec["poi_count_%s_300m" % name] = 0
            rec["restaurant_shp_count_300m"] = 0
            rows.append(rec)
            continue
        poly = geom.buffer(BUFFER_M)
        idxs = strtree_query_indices(poi_tree, poi_geoms, poi_wkb, poly)
        labels = poi_major[idxs] if len(idxs) > 0 else []
        _, ent_norm = shannon_entropy(labels, global_K=global_K)
        rec = {"conedge_id": cid, "poi_count_300m": int(len(idxs)), "poi_mix_entropy_norm_300m": float(ent_norm)}
        for name, mask in target_masks.items():
            rec["poi_count_%s_300m" % name] = int(np.sum(mask[idxs])) if len(idxs) > 0 else 0
        if rest_tree is not None:
            ridxs = strtree_query_indices(rest_tree, rest_geoms, rest_wkb, poly)
            rec["restaurant_shp_count_300m"] = int(len(ridxs))
        else:
            rec["restaurant_shp_count_300m"] = np.nan
        rows.append(rec)
    return pd.DataFrame(rows)


def get_or_build_landuse(cache):
    req = ["conedge_id", "poi_count_300m", "poi_mix_entropy_norm_300m"] + ["poi_count_%s_300m" % name for name in POI_TARGET_KEYWORDS.keys()]
    if should_reuse_file(LANDUSE_CONEDGE_CSV, force=FORCE_REBUILD_LANDUSE_FEATURES, required_columns=req):
        print("[reuse] landuse features")
        return pd.read_csv(LANDUSE_CONEDGE_CSV)
    land = build_conedge_landuse(cache)
    write_csv_atomic(land, LANDUSE_CONEDGE_CSV, index=False, compression="gzip")
    return land


## 8. Aggregate features to segment level

Road class, network structure, and POI features are aggregated to each movement segment by traversed edge length.


In [ ]:
# =========================================================
# 8. Segment level aggregation
# =========================================================
def build_conedge_feature_matrix(cache, landuse_df):
    gdf_con = cache["gdf_con"].copy()
    if landuse_df is not None and len(landuse_df) > 0:
        gdf_con = gdf_con.merge(landuse_df, on="conedge_id", how="left")
    core_cols = [
        "degree_mean_end", "betweenness_raw_mean_end", "closeness_approx_mean_end",
        "tortuosity", "curvature_deg_per_m",
        "road_len_in_300m", "intersection_count_in_300m",
        "connectivity_sum_degree_in_300m", "intersection_density_per_km_300m",
        "poi_count_300m", "poi_mix_entropy_norm_300m", "restaurant_shp_count_300m",
    ]
    for name in POI_TARGET_KEYWORDS.keys():
        core_cols.append("poi_count_%s_300m" % name)
    road_class_cols = [c for c in gdf_con.columns if c.startswith("road_class_share_") or c.startswith("road_class_broad_share_")]
    feature_cols = []
    for c in road_class_cols + core_cols:
        if c in gdf_con.columns and c not in feature_cols:
            feature_cols.append(c)
    df = gdf_con[["conedge_id", "length_m"] + feature_cols].copy()
    for c in feature_cols:
        df[c] = pd.to_numeric(df[c], errors="coerce")
    return df, feature_cols


def aggregate_features_to_segments(segment_df, cache, conedge_feat_df, feature_cols):
    und_len = cache["und_len"]
    und_to_con = cache["und_to_con"]
    conedge_feat_df = conedge_feat_df.set_index("conedge_id")
    conids = conedge_feat_df.index.astype(int).to_numpy()
    conid_to_idx = {int(cid): i for i, cid in enumerate(conids)}
    feat_mat = conedge_feat_df[feature_cols].to_numpy(dtype=float)
    rows = []
    for r in tqdm(segment_df.itertuples(index=False), total=len(segment_df), desc="aggregate to segments"):
        seg_id = int(getattr(r, "seg_id"))
        keys = parse_edge_key_list(getattr(r, "path_edge_keys", ""))
        acc = defaultdict(float)
        total_len = 0.0
        for key in keys:
            L = float(und_len.get(key, np.nan))
            cid = und_to_con.get(key, None)
            if cid is None or not np.isfinite(L) or L <= 0:
                continue
            acc[int(cid)] += L
            total_len += L
        row = {"seg_id": seg_id, "path_len_m_from_edges": float(total_len) if total_len > 0 else np.nan, "n_conedges_in_segment": int(len(acc))}
        if total_len > 0 and len(acc) > 0:
            vec = np.zeros(len(feature_cols), dtype=float)
            used_weight = 0.0
            for cid, w in acc.items():
                idx = conid_to_idx.get(int(cid), None)
                if idx is None:
                    continue
                vals = feat_mat[idx]
                vals2 = vals.copy()
                vals2[np.isnan(vals2)] = 0.0
                vec += float(w) * vals2
                used_weight += float(w)
            if used_weight > 0:
                vec = vec / used_weight
                for j, c in enumerate(feature_cols):
                    row[c + "_lenw_mean"] = float(vec[j])
        rows.append(row)
    return pd.DataFrame(rows)


## 9. VIF and final table layout

Road class columns are now dynamic because they come from the current GraphML. Add new variable names to the static lists below only when they are not generated by prefix.


In [6]:
# =========================================================
# 9. VIF and final table layout
# =========================================================
ID_COLS = ["seg_id", "parent_route_id", "segment_index", "dt", "courier_id", "wave_id"]
TRIP_FEATURE_COLS = [
    "duration", "final_distance_m", "straight_distance_m", "speed_kmh",
    "time_pressure_sec", "time_pressure_min", "onhand_order_count_start",
    "start_hour_local", "start_hour_sin", "start_hour_cos", "start_weekday_local",
    "is_weekend_local", "is_workday_local",
]
RIDER_FEATURE_COLS = [
    "rider_full_time_like", "rider_works_all_7_weekdays", "rider_active_days",
    "rider_active_day_share_in_data", "rider_avg_orders_per_active_day",
    "rider_total_orders_raw", "rider_mean_onhand_raw", "rider_median_onhand_raw",
    "rider_share_event_onhand_ge2_raw", "rider_share_event_onhand_ge3_raw",
    "rider_mean_max_onhand_per_wave", "rider_median_max_onhand_per_wave",
    "rider_share_single_order_wave", "rider_share_batch_wave_ge2", "rider_share_batch_wave_ge3",
    "rider_share_grab_when_already_onhand",
    "rider_mean_segment_distance_m", "rider_median_segment_distance_m", "rider_share_long_segment",
]

# Explicit GraphML road class variables. These are fixed from baoding_clear.graphml diagnostics.
GRAPHML_ROAD_CLASS_FEATURE_COLS = [
    "road_class_share_levelFourRoad_lenw_mean",
    "road_class_share_levelThreeRoad_lenw_mean",
    "road_class_share_secondaryRoad_lenw_mean",
    "road_class_share_nationalRoad_lenw_mean",
    "road_class_share_provincialRoad_lenw_mean",
    "road_class_share_overPass_lenw_mean",
    "road_class_share_unknown_lenw_mean",
]
# Broad road-class variables are generated and kept available, but not used by default when detailed classes are used.
GRAPHML_ROAD_CLASS_BROAD_FEATURE_COLS = [
    "road_class_broad_share_level_four_lenw_mean",
    "road_class_broad_share_level_three_lenw_mean",
    "road_class_broad_share_secondary_lenw_mean",
    "road_class_broad_share_national_or_provincial_lenw_mean",
    "road_class_broad_share_overpass_lenw_mean",
    "road_class_broad_share_unknown_lenw_mean",
]

STATIC_ROAD_FEATURE_COLS = [
    "path_len_m_from_edges", "n_conedges_in_segment",
    "degree_mean_end_lenw_mean", "betweenness_raw_mean_end_lenw_mean",
    "closeness_approx_mean_end_lenw_mean", "tortuosity_lenw_mean",
    "curvature_deg_per_m_lenw_mean", "road_len_in_300m_lenw_mean",
    "intersection_count_in_300m_lenw_mean", "connectivity_sum_degree_in_300m_lenw_mean",
    "intersection_density_per_km_300m_lenw_mean",
]
STATIC_LANDUSE_FEATURE_COLS = [
    "poi_count_300m_lenw_mean",
    "poi_mix_entropy_norm_300m_lenw_mean",
    "poi_count_restaurant_300m_lenw_mean",
    "poi_count_traffic_facility_300m_lenw_mean",
    "poi_count_passage_facility_300m_lenw_mean",
    "poi_count_residential_business_300m_lenw_mean",
    "poi_count_company_300m_lenw_mean",
    "poi_count_government_300m_lenw_mean",
    "poi_count_finance_300m_lenw_mean",
    "poi_count_education_culture_300m_lenw_mean",
    "poi_count_accommodation_300m_lenw_mean",
    "poi_count_public_facility_300m_lenw_mean",
    "poi_count_scenic_300m_lenw_mean",
    "poi_count_shopping_300m_lenw_mean",
    "poi_count_life_service_300m_lenw_mean",
    "restaurant_shp_count_300m_lenw_mean",
]
OUTCOME_COLS = ["overspeed_20", "speed_kmh"]
AUX_COLS = [
    "start_time", "end_time", "start_ts_local", "start_date_local",
    "start_action", "end_action", "start_order_id", "end_order_id",
    "start_lng", "start_lat", "end_lng", "end_lat",
    "dist_mode", "calc_method", "path_edge_keys",
]


def explicit_road_class_cols(df, include_broad=False):
    cols = [c for c in GRAPHML_ROAD_CLASS_FEATURE_COLS if c in df.columns]
    if include_broad:
        cols = cols + [c for c in GRAPHML_ROAD_CLASS_BROAD_FEATURE_COLS if c in df.columns]
    return cols


def dynamic_road_class_cols(df):
    # Kept for backward compatibility with older cells.
    return explicit_road_class_cols(df, include_broad=False)


def add_outcome_columns(df):
    out = df.copy()
    if "speed_kmh" not in out.columns:
        out["speed_kmh"] = pd.to_numeric(out["final_distance_m"], errors="coerce") / pd.to_numeric(out["duration"], errors="coerce") * 3.6
    out["overspeed_20"] = (pd.to_numeric(out["speed_kmh"], errors="coerce") > OVERSPEED_THRESHOLD_KMH).astype(int)
    return out


def build_variable_dictionary(df):
    road_class_cols = dynamic_road_class_cols(df)
    specs = [
        ("id", "ids", ID_COLS),
        ("x", "trip", TRIP_FEATURE_COLS),
        ("x", "rider", RIDER_FEATURE_COLS),
        ("x", "road_class_graphml", road_class_cols),
        ("x", "road_network", STATIC_ROAD_FEATURE_COLS),
        ("x", "landuse", STATIC_LANDUSE_FEATURE_COLS),
        ("y", "outcomes", OUTCOME_COLS),
        ("aux", "auxiliary", AUX_COLS),
    ]
    return reorder_with_groups(df, specs, table_name="segment_model")


def drop_near_duplicate_columns(X, threshold=0.995):
    cols = list(X.columns)
    keep = []
    dropped = []
    for c in cols:
        if len(keep) == 0:
            keep.append(c)
            continue
        vals = []
        for k in keep:
            try:
                corr = float(np.corrcoef(X[c].values, X[k].values)[0, 1])
            except Exception:
                corr = np.nan
            if np.isfinite(corr):
                vals.append((k, abs(corr)))
        if len(vals) > 0:
            best_col, best_corr = max(vals, key=lambda z: z[1])
            if best_corr >= threshold:
                dropped.append({"column_name": c, "dropped_reason": "high_pairwise_corr", "corr_with": best_col, "abs_corr": best_corr})
                continue
        keep.append(c)
    return X[keep].copy(), pd.DataFrame(dropped)


def compute_vif_table(df, candidate_cols):
    cols = [c for c in candidate_cols if c in df.columns]
    X = df[cols].apply(pd.to_numeric, errors="coerce")
    non_null_share = X.notna().mean()
    cols = [c for c in X.columns if non_null_share[c] >= VIF_MIN_NON_NULL_SHARE]
    X = X[cols].copy()
    good = []
    for c in X.columns:
        if X[c].nunique(dropna=True) > 1:
            good.append(c)
    X = X[good]
    if len(X.columns) == 0:
        return pd.DataFrame(columns=["column_name", "vif", "r_squared", "n_rows", "note"])
    if len(X) > VIF_MAX_ROWS:
        X = X.sample(n=VIF_MAX_ROWS, random_state=RANDOM_SEED)
    X = X.fillna(X.median(numeric_only=True))
    X = X.replace([np.inf, -np.inf], np.nan).fillna(0.0)
    X = (X - X.mean()) / X.std(ddof=0).replace(0, np.nan)
    X = X.replace([np.inf, -np.inf], np.nan).fillna(0.0)
    X, dropped_corr = drop_near_duplicate_columns(X, threshold=VIF_CORR_DROP_THRESHOLD)
    arr = X.to_numpy(dtype=float)
    n, p = arr.shape
    rows = []
    for j, c in enumerate(tqdm(X.columns, desc="VIF variables")):
        y = arr[:, j]
        others = np.delete(arr, j, axis=1)
        if others.shape[1] == 0:
            rows.append({"column_name": c, "vif": 1.0, "r_squared": 0.0, "n_rows": int(n), "note": "only_variable"})
            continue
        Xreg = np.column_stack([np.ones(n), others])
        try:
            beta, _, _, _ = np.linalg.lstsq(Xreg, y, rcond=None)
            yhat = Xreg.dot(beta)
            sse = float(np.sum((y - yhat) ** 2))
            sst = float(np.sum((y - y.mean()) ** 2))
            r2 = 1.0 - sse / sst if sst > 1e-12 else 0.0
            r2 = min(max(r2, 0.0), 0.999999)
            vif = 1.0 / max(1.0 - r2, 1e-6)
            note = "ok"
        except Exception as exc:
            r2 = np.nan
            vif = np.nan
            note = "failed: %s" % exc
        rows.append({"column_name": c, "vif": float(vif), "r_squared": float(r2), "n_rows": int(n), "note": note})
    out = pd.DataFrame(rows).sort_values("vif", ascending=False, na_position="last")
    if len(dropped_corr) > 0:
        add = dropped_corr.copy()
        add["vif"] = np.nan
        add["r_squared"] = np.nan
        add["n_rows"] = int(n)
        add["note"] = add.apply(lambda r: "dropped before VIF, corr with %s = %.4f" % (r["corr_with"], r["abs_corr"]), axis=1)
        out = pd.concat([out, add[["column_name", "vif", "r_squared", "n_rows", "note"]]], ignore_index=True)
    return out


## 10. Run pipeline

This cell runs all stages. With default settings, existing expensive graph cache is reused, road class columns are refreshed from GraphML, and only missing downstream files are rebuilt.


In [ ]:
# =========================================================
# 10. Run pipeline
# =========================================================
def load_segment_base():
    df = pd.read_csv(SEGMENT_CSV)
    df = df.reset_index(drop=True)
    if "seg_id" not in df.columns:
        df["seg_id"] = np.arange(len(df), dtype=np.int64)
    for c in ["dt", "courier_id", "wave_id", "start_time", "end_time", "duration", "final_distance_m", "straight_distance_m", "speed_kmh"]:
        if c in df.columns:
            df[c] = pd.to_numeric(df[c], errors="coerce")
    if "speed_kmh" not in df.columns:
        df["speed_kmh"] = df["final_distance_m"] / df["duration"].replace(0, np.nan) * 3.6
    if "time_pressure_min" not in df.columns and "time_pressure_sec" in df.columns:
        df["time_pressure_min"] = pd.to_numeric(df["time_pressure_sec"], errors="coerce") / 60.0
    df = add_outcome_columns(df)
    df = add_time_columns(df, start_col="start_time")
    return df


def main():
    print("Input segment file:", SEGMENT_CSV)
    seg = load_segment_base()
    print("Segment rows:", f"{len(seg):,}")
    if "path_edge_keys" not in seg.columns:
        path_keys, schema = build_segment_edge_keys(seg)
        seg["path_edge_keys"] = path_keys
    else:
        schema = "existing_path_edge_keys"
    print("Path schema:", schema)

    event_state = get_or_build_event_state()
    seg = attach_event_state_to_segments(seg, event_state)
    rider = get_or_build_rider_features(event_state, seg)
    seg = seg.merge(rider, on="courier_id", how="left")

    cache = get_or_build_graph_features()
    land = get_or_build_landuse(cache)
    conedge_feat, feature_cols = build_conedge_feature_matrix(cache, land)
    seg_agg = aggregate_features_to_segments(seg, cache, conedge_feat, feature_cols)
    seg_model = seg.merge(seg_agg, on="seg_id", how="left")

    seg_model, var_dict = build_variable_dictionary(seg_model)
    write_csv_atomic(seg_model, SEGMENT_MODEL_CSV_GZ, index=False, compression="gzip")
    if WRITE_CSV_COPY:
        write_csv_atomic(seg_model, SEGMENT_MODEL_CSV, index=False)
    write_csv_atomic(var_dict, VARIABLE_DICTIONARY_CSV, index=False)

    road_class_cols = dynamic_road_class_cols(seg_model)
    candidate_cols = []
    candidate_cols.extend(TRIP_FEATURE_COLS)
    candidate_cols.extend(RIDER_FEATURE_COLS)
    candidate_cols.extend(road_class_cols)
    candidate_cols.extend(STATIC_ROAD_FEATURE_COLS)
    candidate_cols.extend(STATIC_LANDUSE_FEATURE_COLS)
    if should_reuse_file(VIF_CSV, force=FORCE_REBUILD_VIF, required_columns=["column_name", "vif"]):
        print("[reuse] VIF table")
        vif = pd.read_csv(VIF_CSV)
    else:
        vif = compute_vif_table(seg_model, candidate_cols)
        write_csv_atomic(vif, VIF_CSV, index=False)
    print("Final segment model table:", SEGMENT_MODEL_CSV_GZ)
    print("Variable dictionary:", VARIABLE_DICTIONARY_CSV)
    print("VIF table:", VIF_CSV)
    return seg_model, var_dict, vif

segment_model, variable_dictionary, vif_table = main()
segment_model.head()


## 11. Quick checks

Use these outputs to verify GraphML road classes, missingness, and high VIF variables.


In [ ]:
# =========================================================
# 11. Quick checks
# =========================================================
print("Rows:", f"{len(segment_model):,}")
print("Columns:", f"{segment_model.shape[1]:,}")
print("GraphML road class values")
display(road_class_values.head(30))

road_cols = dynamic_road_class_cols(segment_model)
print("Explicit road class variables:", len(road_cols))
print(road_cols)

summary_cols = [
    "speed_kmh", "duration", "final_distance_m", "time_pressure_min",
    "onhand_order_count_start", "rider_avg_orders_per_active_day",
    "degree_mean_end_lenw_mean", "intersection_density_per_km_300m_lenw_mean",
    "poi_mix_entropy_norm_300m_lenw_mean",
] + road_cols[:8]
summary_cols = [c for c in summary_cols if c in segment_model.columns]
display(segment_model[summary_cols].describe(percentiles=[0.01, 0.05, 0.5, 0.95, 0.99]).T)

display(vif_table.head(30))


## 12. Correlation and VIF control panel

Use this block after the final segment table is created. It first reports highly correlated numeric variable pairs. Then edit `MANUAL_DROP_VARIABLES_FOR_VIF` to remove variables you do not want in the diagnostic VIF table. The block writes a new VIF table and a selected-variable list.


In [12]:
# =========================================================
# 12. Correlation and VIF control panel
#    Edit MANUAL_DROP_VARIABLES_FOR_VIF, then rerun this cell.
# =========================================================
from pathlib import Path
import numpy as np
import pandas as pd

try:
    CORR_VIF_DIR = OUTPUT_ROOT / "diagnostics"
except NameError:
    CORR_VIF_DIR = Path("./outputs_new_clear/segment_variables_graphml_roadclass/diagnostics")
CORR_VIF_DIR.mkdir(parents=True, exist_ok=True)

HIGH_CORR_PAIRS_CSV = CORR_VIF_DIR / "high_correlation_pairs.csv"
MANUAL_VIF_CSV = CORR_VIF_DIR / "vif_after_manual_drops.csv"
SELECTED_VARIABLES_CSV = CORR_VIF_DIR / "selected_variables_after_manual_drops.csv"
MODEL_VARIABLE_BLOCK_TXT = CORR_VIF_DIR / "model_variable_block_for_copy.py"

# Change this threshold if the pair table is too long or too short.
HIGH_CORR_THRESHOLD = 0.5

# Add variable names here, one per line. Keep the comma.
# Example:
# MANUAL_DROP_VARIABLES_FOR_VIF = [
#     "road_class_share_unknown_lenw_mean",
#     "poi_count_300m_lenw_mean",
# ]
MANUAL_DROP_VARIABLES_FOR_VIF = [
    # -----------------------------
    # 1. Target leakage or target components
    # -----------------------------
    "speed_kmh",
    "final_distance_m",
    "duration",

    # -----------------------------
    # 2. Exact duplicates or exact complements
    # -----------------------------
    "time_pressure_sec",
    "is_workday_local",
    "rider_share_single_order_wave",
    "rider_active_days",

    # -----------------------------
    # 3. Distance/path length duplicates
    # -----------------------------
    "path_len_m_from_edges",
    # "straight_distance_m",

    # -----------------------------
    # 4. Local road network density cluster
    # Keep intersection_density_per_km_300m_lenw_mean
    # -----------------------------
    "intersection_count_in_300m_lenw_mean",
    "connectivity_sum_degree_in_300m_lenw_mean",

    # -----------------------------
    # 5. Rider batching behavior cluster
    # Keep rider_share_batch_wave_ge2 and onhand_order_count_start
    # -----------------------------
    "rider_mean_onhand_raw",
    "rider_share_event_onhand_ge2_raw",
    "rider_share_event_onhand_ge3_raw",
    "rider_mean_max_onhand_per_wave",
    "rider_share_grab_when_already_onhand",

    # -----------------------------
    # 6. Rider productivity cluster
    # Keep rider_avg_orders_per_active_day
    # -----------------------------
    "rider_total_orders_raw",

    # -----------------------------
    # 7. Rider distance preference cluster
    # Keep rider_median_segment_distance_m
    # -----------------------------
    "rider_mean_segment_distance_m",
    "rider_share_long_segment",

    # Path length and route-complexity cluster
    "n_conedges_in_segment",

    # Time encoding cluster
    "start_hour_local",
    "start_weekday_local",

    # Rider activity intensity cluster
    "rider_full_time_like",
    "rider_works_all_7_weekdays",

    # Rider batching cluster
    "rider_share_batch_wave_ge3",
]

# Road-class shares sum to one, so one reference column should be excluded for models with an intercept.
ROAD_CLASS_REFERENCE_FOR_VIF = "road_class_share_unknown_lenw_mean"


def get_current_segment_model_table():
    if "segment_model" in globals():
        return segment_model.copy()
    candidates = []
    for name in ["SEGMENT_MODEL_CSV_GZ", "SEGMENT_MODEL_CSV"]:
        if name in globals():
            candidates.append(globals()[name])
    candidates.extend([
        Path("./outputs_new_clear/segment_variables_graphml_roadclass/segments/segment_model_table.csv.gz"),
        Path("./outputs_new_clear/segment_variables_graphml_roadclass/segments/segment_model_table.csv"),
        Path("./outputs_new_clear/segment_variables_clean/segments/segment_model_table.csv.gz"),
        Path("./outputs_new_clear/segment_variables_clean/segments/segment_model_table.csv"),
    ])
    for p in candidates:
        p = Path(p)
        if p.exists():
            print("Load segment model table:", p)
            return pd.read_csv(p)
    raise FileNotFoundError("Cannot find segment_model table.")


def existing(cols, df):
    return [c for c in cols if c in df.columns]


def get_default_candidate_groups(df):
    groups = {
        "trip_state": existing(TRIP_FEATURE_COLS, df),
        "rider_behavior": existing(RIDER_FEATURE_COLS, df),
        "road_class_shares": existing(GRAPHML_ROAD_CLASS_FEATURE_COLS, df),
        "road_topology_geometry": existing(STATIC_ROAD_FEATURE_COLS, df),
        "landuse": existing(STATIC_LANDUSE_FEATURE_COLS, df),
    }
    return groups


def flatten_groups(groups):
    out = []
    for g, cols in groups.items():
        for c in cols:
            if c not in out:
                out.append(c)
    return out


def high_corr_pairs(df, cols, threshold=0.85, max_rows=200000):
    cols = [c for c in cols if c in df.columns]
    X = df[cols].apply(pd.to_numeric, errors="coerce")
    keep = []
    for c in X.columns:
        if X[c].notna().mean() >= 0.80 and X[c].nunique(dropna=True) > 1:
            keep.append(c)
    X = X[keep]
    if len(X) > max_rows:
        X = X.sample(n=max_rows, random_state=RANDOM_SEED)
    X = X.replace([np.inf, -np.inf], np.nan)
    corr = X.corr().abs()
    rows = []
    cols2 = list(corr.columns)
    for i in range(len(cols2)):
        for j in range(i + 1, len(cols2)):
            v = corr.iloc[i, j]
            if pd.notna(v) and float(v) >= threshold:
                rows.append({"var1": cols2[i], "var2": cols2[j], "abs_corr": float(v)})
    out = pd.DataFrame(rows)
    if len(out) > 0:
        out = out.sort_values("abs_corr", ascending=False).reset_index(drop=True)
    return out


def compute_vif_manual(df, cols, max_rows=200000):
    cols = [c for c in cols if c in df.columns]
    X = df[cols].apply(pd.to_numeric, errors="coerce")
    keep = []
    for c in X.columns:
        if X[c].notna().mean() >= VIF_MIN_NON_NULL_SHARE and X[c].nunique(dropna=True) > 1:
            keep.append(c)
    X = X[keep]
    if len(X) > max_rows:
        X = X.sample(n=max_rows, random_state=RANDOM_SEED)
    X = X.replace([np.inf, -np.inf], np.nan)
    X = X.fillna(X.median(numeric_only=True)).fillna(0.0)
    X = (X - X.mean()) / X.std(ddof=0).replace(0, np.nan)
    X = X.replace([np.inf, -np.inf], np.nan).fillna(0.0)
    arr = X.to_numpy(dtype=float)
    n = arr.shape[0]
    rows = []
    for j, c in enumerate(X.columns):
        y = arr[:, j]
        others = np.delete(arr, j, axis=1)
        if others.shape[1] == 0:
            rows.append({"column_name": c, "vif": 1.0, "r_squared": 0.0, "n_rows": int(n), "note": "only_variable"})
            continue
        Xreg = np.column_stack([np.ones(n), others])
        try:
            beta, _, _, _ = np.linalg.lstsq(Xreg, y, rcond=None)
            yhat = Xreg.dot(beta)
            sse = float(np.sum((y - yhat) ** 2))
            sst = float(np.sum((y - y.mean()) ** 2))
            r2 = 1.0 - sse / sst if sst > 1e-12 else 0.0
            r2 = min(max(r2, 0.0), 0.999999)
            vif = 1.0 / max(1.0 - r2, 1e-6)
            note = "ok"
        except Exception as exc:
            r2 = np.nan
            vif = np.nan
            note = "failed: %s" % exc
        rows.append({"column_name": c, "vif": float(vif), "r_squared": float(r2), "n_rows": int(n), "note": note})
    return pd.DataFrame(rows).sort_values("vif", ascending=False, na_position="last").reset_index(drop=True)


df_for_vif = get_current_segment_model_table()
candidate_groups = get_default_candidate_groups(df_for_vif)
all_candidates = flatten_groups(candidate_groups)

initial_corr_pairs = high_corr_pairs(df_for_vif, all_candidates, threshold=HIGH_CORR_THRESHOLD, max_rows=VIF_MAX_ROWS)
initial_corr_pairs.to_csv(HIGH_CORR_PAIRS_CSV, index=False)
print("High-correlation pairs saved:", HIGH_CORR_PAIRS_CSV)
print("High-correlation pairs:", len(initial_corr_pairs))
display(initial_corr_pairs.head(100))

manual_drop = set(MANUAL_DROP_VARIABLES_FOR_VIF)
if ROAD_CLASS_REFERENCE_FOR_VIF:
    manual_drop.add(ROAD_CLASS_REFERENCE_FOR_VIF)

selected_groups = {}
for group_name, cols in candidate_groups.items():
    selected_groups[group_name] = [c for c in cols if c not in manual_drop]
selected_candidates = flatten_groups(selected_groups)

selected_df = pd.DataFrame([
    {"group_name": g, "variable": c, "selected": int(c in selected_candidates), "dropped": int(c in manual_drop)}
    for g, cols in candidate_groups.items()
    for c in cols
])
selected_df.to_csv(SELECTED_VARIABLES_CSV, index=False)

manual_vif = compute_vif_manual(df_for_vif, selected_candidates, max_rows=VIF_MAX_ROWS)
manual_vif.to_csv(MANUAL_VIF_CSV, index=False)
print("Selected variables saved:", SELECTED_VARIABLES_CSV)
print("Manual VIF saved:", MANUAL_VIF_CSV)
print("Selected candidate count:", len(selected_candidates))
display(manual_vif.head(80))


Load segment model table: c:\Users\Kwk10\Desktop\2026 Spring\Meituan\outputs_new_clear\segment_variables_clean\segments\segment_model_table.csv.gz
High-correlation pairs saved: c:\Users\Kwk10\Desktop\2026 Spring\Meituan\outputs_new_clear\segment_variables_clean\diagnostics\high_correlation_pairs.csv
High-correlation pairs: 122


,var1,var2,abs_corr
0,rider_share_single_order_wave,rider_share_batch_wave_ge2,1.000000
1,rider_active_days,rider_active_day_share_in_data,1.000000
2,is_weekend_local,is_workday_local,1.000000
3,time_pressure_sec,time_pressure_min,1.000000
4,final_distance_m,path_len_m_from_edges,0.997428
...,...,...,...
95,rider_share_single_order_wave,rider_mean_segment_distance_m,0.570648
96,rider_share_batch_wave_ge2,rider_mean_segment_distance_m,0.570648
97,rider_avg_orders_per_active_day,rider_share_long_segment,0.570078
98,rider_mean_onhand_raw,rider_share_long_segment,0.567368


Selected variables saved: c:\Users\Kwk10\Desktop\2026 Spring\Meituan\outputs_new_clear\segment_variables_clean\diagnostics\selected_variables_after_manual_drops.csv
Manual VIF saved: c:\Users\Kwk10\Desktop\2026 Spring\Meituan\outputs_new_clear\segment_variables_clean\diagnostics\vif_after_manual_drops.csv
Selected candidate count: 29


,column_name,vif,r_squared,n_rows,note
0,poi_count_300m_lenw_mean,4.366062,0.770961,200000,ok
1,poi_mix_entropy_norm_300m_lenw_mean,3.774075,0.735034,200000,ok
2,closeness_approx_mean_end_lenw_mean,3.587203,0.721231,200000,ok
3,rider_share_batch_wave_ge2,3.335702,0.700213,200000,ok
4,rider_median_max_onhand_per_wave,2.869156,0.651465,200000,ok
5,restaurant_shp_count_300m_lenw_mean,2.360099,0.576289,200000,ok
6,rider_median_onhand_raw,2.197056,0.544846,200000,ok
7,road_len_in_300m_lenw_mean,2.073850,0.517805,200000,ok
8,rider_median_segment_distance_m,2.030250,0.507450,200000,ok
9,intersection_density_per_km_300m_lenw_mean,1.689377,0.408066,200000,ok


## 13. Copy-paste variable block for the model notebook

After you are satisfied with the manual drop list and VIF table, run this block. It prints variables in the same `X_GROUPS = OrderedDict([...])` style used by the model notebook. Copy the printed block into the model notebook configuration cell.


In [13]:
# =========================================================
# 13. Print selected variables in model-notebook format
# =========================================================
def format_python_list(items, indent="        "):
    if len(items) == 0:
        return "[]"
    lines = ["["]
    for x in items:
        lines.append('%s"%s",' % (indent, x))
    lines.append("    ]")
    return "\n".join(lines)

if "selected_groups" not in globals():
    raise RuntimeError("Run the correlation and VIF control panel first.")

lines = []
lines.append("X_GROUPS = OrderedDict([")
for group_name in ["trip_state", "rider_behavior", "road_class_shares", "road_topology_geometry", "landuse"]:
    cols = selected_groups.get(group_name, [])
    lines.append('    ("%s", %s),' % (group_name, format_python_list(cols)))
lines.append("])")

model_block = "\n".join(lines)
print(model_block)

MODEL_VARIABLE_BLOCK_TXT.write_text(model_block, encoding="utf-8")
print("Saved copy-paste block:", MODEL_VARIABLE_BLOCK_TXT)


X_GROUPS = OrderedDict([
    ("trip_state", [
        "straight_distance_m",
        "time_pressure_min",
        "onhand_order_count_start",
        "start_hour_sin",
        "start_hour_cos",
        "is_weekend_local",
    ]),
    ("rider_behavior", [
        "rider_active_day_share_in_data",
        "rider_avg_orders_per_active_day",
        "rider_median_onhand_raw",
        "rider_median_max_onhand_per_wave",
        "rider_share_batch_wave_ge2",
        "rider_median_segment_distance_m",
    ]),
    ("road_class_shares", [
        "road_class_share_levelFourRoad_lenw_mean",
        "road_class_share_levelThreeRoad_lenw_mean",
        "road_class_share_secondaryRoad_lenw_mean",
        "road_class_share_nationalRoad_lenw_mean",
        "road_class_share_provincialRoad_lenw_mean",
        "road_class_share_overPass_lenw_mean",
    ]),
    ("road_topology_geometry", [
        "degree_mean_end_lenw_mean",
        "betweenness_raw_mean_end_lenw_mean",
        "closeness_approx_mean_e